# EAGF Notebook 2: Statistical Analysis

This notebook performs and explains all statistical tests used in the paper:
- Two-proportion z-test (accuracy comparison)
- Paired t-test / Wilcoxon signed-rank (Trust Index comparison)
- Bootstrap confidence intervals (all pillar metrics)
- Ablation significance across all six variants

**Paper reference:** Section 5.1.3 (Statistical Analysis Protocol)

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
!git clone https://github.com/aliakarma/eagf.git
!cd eafg

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yaml

print('Imports ready.')

## 1. Run 3-Seed Experiment

In [ ]:
import sys, os
PROJECT_ROOT = '/content/eagf'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.data_loader import generate_demo_biometric
from src.training.eagf_trainer import train_variant

import yaml # yaml is used below and implicitly assumed to be imported earlier

with open(os.path.join(PROJECT_ROOT, 'configs', 'biometric_default.yaml')) as f:
    config = yaml.safe_load(f)
config['training']['epochs'] = 30

SEEDS    = [42, 123, 456]
VARIANTS = ['baseline', 'transparency', 'fairness', 'privacy', 'accountability', 'eagf']
LABELS   = {
    'baseline': 'M0: Baseline', 'transparency': 'M1: +Transparency',
    'fairness': 'M2: +Fairness', 'privacy': 'M3: +Privacy',
    'accountability': 'M4: +Accountability', 'eagf': 'M5: EAGF (Full)',
}

dataset = generate_demo_biometric(n_samples=1200, seed=42)

print(f'Running {len(VARIANTS)} variants × {len(SEEDS)} seeds = {len(VARIANTS)*len(SEEDS)} runs...')
all_results = {v: [] for v in VARIANTS}
for v in VARIANTS:
    for seed in SEEDS:
        m = train_variant(v, config, dataset.copy(), seed=seed,
                          output_dir=f'/tmp/eagf_nb2/{v}/seed_{seed}')
        all_results[v].append(m)
print('Done.')

## 2. Bootstrap Confidence Intervals

In [ ]:
from src.evaluation.statistics import bootstrap_ci
import pandas as pd

METRICS = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'accountability', 'trust_index']
rows = []
for v in VARIANTS:
    row = {'Model': LABELS[v]}
    for m in METRICS:
        vals = np.array([r[m] for r in all_results[v]])
        ci = bootstrap_ci(vals, n_resamples=1000)
        row[m] = f"{ci['mean']:.3f} [{ci['ci_lower']:.3f}, {ci['ci_upper']:.3f}]"
    rows.append(row)

df_ci = pd.DataFrame(rows).set_index('Model')
print('Mean [95% Bootstrap CI] across 3 seeds:')
print('=' * 100)
print(df_ci.to_string())

## 3. Accuracy: Two-Proportion Z-Test (Baseline vs. EAGF)

In [ ]:
from src.evaluation.statistics import two_proportion_ztest

acc_baseline = np.mean([r['accuracy'] for r in all_results['baseline']])
acc_eagf     = np.mean([r['accuracy'] for r in all_results['eagf']])
n_test       = len(dataset['y_test'])

result = two_proportion_ztest(acc_eagf, acc_baseline, n_test, n_test)

print('Two-Proportion Z-Test: Accuracy (EAGF vs. Baseline)')
print('=' * 55)
print(f'  Baseline accuracy : {acc_baseline:.4f}')
print(f'  EAGF accuracy     : {acc_eagf:.4f}')
print(f'  Difference        : {acc_eagf - acc_baseline:+.4f}')
print(f'  Z-statistic       : {result["z"]:+.4f}')
print(f'  p-value           : {result["p_value"]:.4f}')
interp = '✓ Not significant (p > 0.05) — acceptable accuracy trade-off' \
    if result['p_value'] > 0.05 else '✗ Significant (p ≤ 0.05)'
print(f'  Interpretation    : {interp}')

## 4. Trust Index: Paired T-Test (Baseline vs. EAGF)

In [ ]:
from scipy.stats import ttest_rel

ti_baseline = np.array([r['trust_index'] for r in all_results['baseline']])
ti_eagf     = np.array([r['trust_index'] for r in all_results['eagf']])

stat, pval = ttest_rel(ti_eagf, ti_baseline)

print('Paired T-Test: Trust Index (EAGF vs. Baseline)')
print('=' * 55)
print(f'  Seeds             : {SEEDS}')
print(f'  Baseline TI       : {ti_baseline} (mean={ti_baseline.mean():.3f})')
print(f'  EAGF TI           : {ti_eagf} (mean={ti_eagf.mean():.3f})')
print(f'  Mean difference   : {(ti_eagf - ti_baseline).mean():+.3f}')
print(f'  t-statistic       : {stat:+.4f}')
print(f'  p-value           : {pval:.4f}')
interp = '✓ Significant (p < 0.05)' if pval < 0.05 else 'Not significant (p ≥ 0.05)'
print(f'  Interpretation    : {interp}')
print(f'  Note: Paired t-test used (n={len(SEEDS)} < 5 required for Wilcoxon)')

## 5. Confidence Interval Plot — All Variants

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours   = ['#9E9E9E','#42A5F5','#66BB6A','#EF5350','#FFA726','#2E7D32']

for ax, metric, ylabel in zip(
    axes,
    ['recall_parity', 'trust_index'],
    ['Recall Parity (RP)', 'Trust Index (TI)']
):
    for i, (v, col) in enumerate(zip(VARIANTS, colours)):
        vals = np.array([r[metric] for r in all_results[v]])
        ci   = bootstrap_ci(vals, n_resamples=1000)
        ax.errorbar(
            i, ci['mean'],
            yerr=[[ci['mean'] - ci['ci_lower']], [ci['ci_upper'] - ci['mean']]],
            fmt='o', color=col, capsize=5, markersize=8, linewidth=2,
        )
        ax.text(i, ci['ci_upper'] + 0.005, f"{ci['mean']:.3f}",
                ha='center', fontsize=8, color=col, fontweight='bold')

    short_labels = ['M0','M1','M2','M3','M4','M5']
    ax.set_xticks(range(len(VARIANTS)))
    ax.set_xticklabels(short_labels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f'{ylabel} by Model Variant (mean ± 95% CI)', fontsize=11)
    ax.axhline(1.0, color='grey', linestyle='--', alpha=0.3, label='Ideal')
    ax.grid(alpha=0.2)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out = os.path.join(PROJECT_ROOT, 'figures', 'notebook2_ci_plot.png')
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out}')

## 6. All-Variant TI Significance Matrix

In [ ]:
from scipy.stats import ttest_rel

print('Pairwise Paired T-Test p-values (Trust Index)')
print('Rows = reference, Columns = comparison')
print()

short = ['M0','M1','M2','M3','M4','M5']
print(f'{"":8s}', end='')
for s in short:
    print(f'{s:>8s}', end='')
print()

for i, va in enumerate(VARIANTS):
    print(f'{short[i]:<8s}', end='')
    for j, vb in enumerate(VARIANTS):
        if i == j:
            print(f'{"—":>8s}', end='')
        else:
            a = np.array([r['trust_index'] for r in all_results[va]])
            b = np.array([r['trust_index'] for r in all_results[vb]])
            try:
                _, p = ttest_rel(a, b)
                marker = ' *' if p < 0.05 else '  '
                print(f'{p:.3f}{marker}', end='   ')
            except Exception:
                print(f'{"n/a":>8s}', end='')
    print()

print()
print('  * = significant at p < 0.05')